In [1]:
import numpy as np
import pandas as pd

In [4]:
orig_df = pd.read_csv('../data/csv/clean_df.csv')
clean_df = orig_df.copy()
clean_df.head(1)

,id,href,price,title,status,content,createdAt,boostedAt,user_dbId,user_nickname,...,boost_diff_seconds,is_boosted,title_len,content_len,has_keyword_new,has_keyword_urgent,price_ratio_to_brand,price_ratio_to_label,days_elapsed,target_n_days
0,11121x6p7597,https://www.daangn.com/kr/buy-sell/%EB%A7%88%E...,22000,마리마켓 트렌치자켓 롤업팬츠 세트 S,Ongoing,"마리마켓 트렌치자켓 롤업팬츠 세트\n베이지 색상\n자켓, 팬츠 둘다 사이즈 S\n미...",2025-10-13 11:00:57+00:00,2026-03-03 12:02:21+00:00,24156105,리본,...,12186084.0,1,20,61,0,0,0.223793,0.122495,142.831007,0.0


In [5]:
clean_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 220688 entries, 0 to 220687
Data columns (total 57 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   id                        220688 non-null  str    
 1   href                      220688 non-null  str    
 2   price                     220688 non-null  int64  
 3   title                     220688 non-null  str    
 4   status                    220688 non-null  str    
 5   content                   220688 non-null  str    
 6   createdAt                 220688 non-null  str    
 7   boostedAt                 220688 non-null  str    
 8   user_dbId                 220688 non-null  int64  
 9   user_nickname             220688 non-null  str    
 10  region_name_from_article  220688 non-null  str    
 11  region_id                 220688 non-null  int64  
 12  region_name               220688 non-null  str    
 13  region_in                 220688 non-null  str    
 14 

In [6]:
n_days = 7  # 예: 7일 이내 판매 여부

# 1. 경과 시간(일) 계산
clean_df['days_elapsed'] = (
    clean_df['crawledAt'] - clean_df['createdAt']
).dt.total_seconds() / (24 * 3600)


# 2. Target 변수 생성 함수
def make_target(row, n):
    # status 컬럼의 '거래완료'를 의미하는 정확한 텍스트 확인 필요 (예: 'Completed', 'Sold' 등)
    is_sold = row['status'] != 'Ongoing'  # 혹은 status_detail 사용

    if is_sold and row['days_elapsed'] <= n:
        return 1  # n일 이내 판매됨
    elif not is_sold and row['days_elapsed'] > n:
        return 0  # n일이 지났는데도 안 팔림
    else:
        return np.nan  # 아직 n일이 안 지났는데 안 팔린 상태 (보류)


clean_df['target_n_days'] = clean_df.apply(lambda x: make_target(x, n_days), axis=1)

# 판별 불가능한(보류된) 데이터 학습에서 제외
train_df = clean_df.dropna(subset=['target_n_days']).copy()

TypeError: operation 'sub' not supported for dtype 'str' with dtype 'str'